In [10]:
from pint.models import get_model
from pint.simulation import make_fake_toas_uniform, make_fake_toas_fromtim
from pint.logging import setup as setup_log
from pint.fitter import Fitter
from pint.utils import cmwavex_setup

from io import StringIO
import numpy as np
import astropy.units as u
from copy import deepcopy
from matplotlib import pyplot as plt

In [2]:
setup_log(level="WARNING")

1

In [3]:
par_sim = """
    PSR                                  SIM6
    EPHEM                               DE440
    CLOCK                        TT(BIPM2019)
    UNITS                                 TDB
    RAJ                      5:00:00.00000000 1 
    DECJ                    15:00:00.00000000 1 
    PMRA                                  0.0
    PMDEC                                 0.0
    PX                                    0.0
    F0                                  100.0 1
    F1                                 -1e-15 1
    PEPOCH             55000.0000000000000000
    TNCHROMIDX                              4
    CM				                      1.0 1
    CM1                                  1e-3 1	
    CM2                                  1e-5 1	 
    TNCHROMAMP                          -13.5
    TNCHROMGAM                            4.0
    TNCHROMC                               30
    PLANET_SHAPIRO                          N
    DM                                   15.0 1
    DM1                                  1e-4 1
    TZRMJD             55000.0000000000000000
    TZRSITE                               gbt
    TZRFRQ                             1400.0
    PHOFF                                 0.0 1 0.0
"""

In [4]:
m = get_model(StringIO(par_sim))

In [5]:
ntoas = 2000
toaerrs = np.random.uniform(0.5, 2.0, ntoas) * u.us
freqs = np.linspace(500, 1500, 8) * u.MHz

In [6]:
t = make_fake_toas_uniform(
    startMJD=53001,
    endMJD=57001,
    ntoas=ntoas,
    model=m,
    freq=freqs,
    obs='gbt',
    error=toaerrs,
    add_noise=True,
    add_correlated_noise=True,
    multi_freqs_in_epoch=True,
    name='fake',
    include_bipm=True,
)

In [7]:
m.write_parfile("sim6.par")
t.write_TOA_file("sim6.tim")

In [11]:
m1 = deepcopy(m)
m1.remove_component("PLChromNoise")

Tspan = t.get_mjds().max() - t.get_mjds().min()
cmwavex_setup(m1, Tspan, n_freqs=45)

m1.write_parfile("sim6.wx.par")